In [7]:
import pandas as pd
import sqlite3

# Added isolation_level=None to let us handle transactions manually without conflicts
conn = sqlite3.connect('shopease.db', isolation_level=None)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INT PRIMARY KEY,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    email VARCHAR(100) UNIQUE NOT NULL,
    city VARCHAR(50) NOT NULL,
    state VARCHAR(50) NOT NULL,
    join_date DATE NOT NULL,
    is_premium BOOLEAN DEFAULT FALSE
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    category VARCHAR(50) NOT NULL,
    brand VARCHAR(50) NOT NULL,
    unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    stock_qty INT NOT NULL CHECK (stock_qty >= 0)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INT PRIMARY KEY,
    customer_id INT NOT NULL,
    order_date DATE NOT NULL,
    status VARCHAR(20) NOT NULL DEFAULT 'Pending' CHECK (status IN ('Pending', 'Shipped', 'Delivered', 'Cancelled')),
    total_amount DECIMAL(12,2) NOT NULL CHECK (total_amount >= 0),
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS order_items (
    item_id INT PRIMARY KEY,
    order_id INT NOT NULL,
    product_id INT NOT NULL,
    quantity INT NOT NULL CHECK (quantity > 0),
    unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price > 0),
    discount_pct DECIMAL(5,2) DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100),
    FOREIGN KEY (order_id) REFERENCES orders (order_id),
    FOREIGN KEY (product_id) REFERENCES products (product_id)
);
""")

customers_data = [
    (101, 'Aarav', 'Sharma', 'aarav.s@email.com', 'Mumbai', 'Maharashtra', '2024-01-15', 1),
    (102, 'Priya', 'Patel', 'priya.p@email.com', 'Ahmedabad', 'Gujarat', '2024-02-20', 0),
    (103, 'Rohan', 'Gupta', 'rohan.g@email.com', 'Delhi', 'Delhi', '2024-03-10', 1),
    (104, 'Sneha', 'Reddy', 'sneha.r@email.com', 'Hyderabad', 'Telangana', '2024-04-05', 0),
    (105, 'Vikram', 'Singh', 'vikram.s@email.com', 'Jaipur', 'Rajasthan', '2024-05-12', 1),
    (106, 'Ananya', 'Iyer', 'ananya.i@email.com', 'Chennai', 'Tamil Nadu', '2024-06-18', 0),
    (107, 'Karan', 'Mehta', 'karan.m@email.com', 'Pune', 'Maharashtra', '2024-07-22', 1),
    (108, 'Divya', 'Nair', 'divya.n@email.com', 'Kochi', 'Kerala', '2024-08-30', 0)
]
cursor.executemany("INSERT OR REPLACE INTO customers VALUES (?,?,?,?,?,?,?,?)", customers_data)

products_data = [
    (201, 'Wireless Earbuds', 'Electronics', 'BoAt', 1499.00, 250),
    (202, 'Cotton T-Shirt', 'Clothing', 'Levis', 799.00, 500),
    (203, 'Smart Watch', 'Electronics', 'Noise', 2999.00, 150),
    (204, 'Running Shoes', 'Clothing', 'Nike', 4599.00, 120),
    (205, 'Bluetooth Speaker', 'Electronics', 'JBL', 3499.00, 200),
    (206, 'Bedsheet Set', 'Home', 'Spaces', 1299.00, 300),
    (207, 'Laptop Stand', 'Electronics', 'AmazonBasics', 899.00, 180),
    (208, 'Cushion Covers (Set)', 'Home', 'HomeCenter', 599.00, 400)
]
cursor.executemany("INSERT OR REPLACE INTO products VALUES (?,?,?,?,?,?)", products_data)

orders_data = [
    (1001, 101, '2024-08-01', 'Delivered', 4498.00),
    (1002, 102, '2024-08-03', 'Delivered', 799.00),
    (1003, 103, '2024-08-05', 'Shipped', 7498.00),
    (1004, 101, '2024-08-10', 'Delivered', 3499.00),
    (1005, 104, '2024-08-12', 'Cancelled', 2999.00),
    (1006, 105, '2024-08-15', 'Delivered', 5898.00),
    (1007, 106, '2024-08-18', 'Pending', 1299.00),
    (1008, 103, '2024-08-20', 'Delivered', 899.00),
    (1009, 107, '2024-08-25', 'Shipped', 6098.00),
    (1010, 108, '2024-08-28', 'Delivered', 1598.00)
]
cursor.executemany("INSERT OR REPLACE INTO orders VALUES (?,?,?,?,?)", orders_data)

order_items_data = [
    (5001, 1001, 201, 2, 1499.00, 0),
    (5002, 1001, 207, 1, 899.00, 10),
    (5003, 1002, 202, 1, 799.00, 0),
    (5004, 1003, 203, 1, 2999.00, 0),
    (5005, 1003, 204, 1, 4599.00, 5),
    (5006, 1004, 205, 1, 3499.00, 0),
    (5007, 1005, 203, 1, 2999.00, 0),
    (5008, 1006, 201, 1, 1499.00, 10),
    (5009, 1006, 204, 1, 4599.00, 5),
    (5010, 1007, 206, 1, 1299.00, 0),
    (5011, 1008, 207, 1, 899.00, 0),
    (5012, 1009, 205, 1, 3499.00, 0),
    (5013, 1009, 208, 2, 599.00, 15),
    (5014, 1010, 206, 1, 1299.00, 0),
    (5015, 1010, 208, 1, 599.00, 0)
]
cursor.executemany("INSERT OR REPLACE INTO order_items VALUES (?,?,?,?,?,?)", order_items_data)

print("Database and data loaded perfectly.")

Database and data loaded perfectly.


In [2]:
# Q1: Display all columns and rows from customers
print("--- Q1 Result ---")
display(pd.read_sql_query("SELECT * FROM customers;", conn))

# Q2: Retrieve first_name, last_name, and city
print("\n--- Q2 Result ---")
display(pd.read_sql_query("SELECT first_name, last_name, city FROM customers;", conn))

# Q3: List unique categories
print("\n--- Q3 Result ---")
display(pd.read_sql_query("SELECT DISTINCT category FROM products;", conn))

# Q4: Primary Keys explanation
print("\n--- Q4 Explanation ---")
print("Primary Keys: customers (customer_id), products (product_id), orders (order_id), order_items (item_id).")
print("Why unique and NOT NULL: To serve as a definitive record identifier so data rows never become ambiguous or unreachable.")

# Q5: Email column constraints and handling duplicate inserts
print("\n--- Q5 Explanation ---")
print("Constraints on email: UNIQUE and NOT NULL.")
print("If you insert a duplicate email, the RDBMS aborts the execution throwing a UNIQUE constraint violation error.")

# Q6: Test negative constraint rule
print("\n--- Q6 Test ---")
try:
    cursor.execute("INSERT INTO products VALUES (209, 'Test Product', 'Gadgets', 'BrandX', -50.00, 10);")
except sqlite3.IntegrityError as e:
    print(f"Executed command failed as expected. Error: {e}")

--- Q1 Result ---


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0



--- Q2 Result ---


,first_name,last_name,city
0,Aarav,Sharma,Mumbai
1,Priya,Patel,Ahmedabad
2,Rohan,Gupta,Delhi
3,Sneha,Reddy,Hyderabad
4,Vikram,Singh,Jaipur
5,Ananya,Iyer,Chennai
6,Karan,Mehta,Pune
7,Divya,Nair,Kochi



--- Q3 Result ---


,category
0,Electronics
1,Clothing
2,Home



--- Q4 Explanation ---
Primary Keys: customers (customer_id), products (product_id), orders (order_id), order_items (item_id).
Why unique and NOT NULL: To serve as a definitive record identifier so data rows never become ambiguous or unreachable.

--- Q5 Explanation ---
Constraints on email: UNIQUE and NOT NULL.
If you insert a duplicate email, the RDBMS aborts the execution throwing a UNIQUE constraint violation error.

--- Q6 Test ---
Executed command failed as expected. Error: CHECK constraint failed: unit_price > 0


In [3]:
# Q7: Orders with status = 'Delivered'
print("--- Q7 Result ---")
display(pd.read_sql_query("SELECT * FROM orders WHERE status = 'Delivered';", conn))

# Q8: Electronics with price > 2000
print("\n--- Q8 Result ---")
display(pd.read_sql_query("SELECT * FROM products WHERE category = 'Electronics' AND unit_price > 2000;", conn))

# Q9: Joined in 2024 and from Maharashtra
print("\n--- Q9 Result ---")
display(pd.read_sql_query("SELECT * FROM customers WHERE join_date LIKE '2024%' AND state = 'Maharashtra';", conn))

# Q10: Target date metrics filtering (Not cancelled)
print("\n--- Q10 Result ---")
q10_query = "SELECT * FROM orders WHERE order_date BETWEEN '2024-08-10' AND '2024-08-25' AND status != 'Cancelled';"
display(pd.read_sql_query(q10_query, conn))

# Q11: Index details
print("\n--- Q11 Explanation ---")
print("The index idx_orders_date structures order_date rows like a sorted map, stopping slow full table scans.")
print("Sample Query: SELECT * FROM orders WHERE order_date = '2024-08-15';")

# Q12: SARGable adjustments
print("\n--- Q12 Explanation ---")
print("No, using functions like YEAR(join_date) breaks indexing functionality.")
print("Index-friendly rewrite:")
display(pd.read_sql_query("SELECT * FROM customers WHERE join_date BETWEEN '2024-01-01' AND '2024-12-31';", conn))

--- Q7 Result ---


,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498
1,1002,102,2024-08-03,Delivered,799
2,1004,101,2024-08-10,Delivered,3499
3,1006,105,2024-08-15,Delivered,5898
4,1008,103,2024-08-20,Delivered,899
5,1010,108,2024-08-28,Delivered,1598



--- Q8 Result ---


,product_id,product_name,category,brand,unit_price,stock_qty
0,203,Smart Watch,Electronics,Noise,2999,150
1,205,Bluetooth Speaker,Electronics,JBL,3499,200



--- Q9 Result ---


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1



--- Q10 Result ---


,order_id,customer_id,order_date,status,total_amount
0,1004,101,2024-08-10,Delivered,3499
1,1006,105,2024-08-15,Delivered,5898
2,1007,106,2024-08-18,Pending,1299
3,1008,103,2024-08-20,Delivered,899
4,1009,107,2024-08-25,Shipped,6098



--- Q11 Explanation ---
The index idx_orders_date structures order_date rows like a sorted map, stopping slow full table scans.
Sample Query: SELECT * FROM orders WHERE order_date = '2024-08-15';

--- Q12 Explanation ---
No, using functions like YEAR(join_date) breaks indexing functionality.
Index-friendly rewrite:


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


In [4]:
# Q13: Total orders count
print("--- Q13 Result ---")
display(pd.read_sql_query("SELECT COUNT(*) AS total_orders FROM orders;", conn))

# Q14: Total revenue from 'Delivered' orders
print("\n--- Q14 Result ---")
display(pd.read_sql_query("SELECT SUM(total_amount) AS total_revenue FROM orders WHERE status = 'Delivered';", conn))

# Q15: Average unit_price of products per category
print("\n--- Q15 Result ---")
display(pd.read_sql_query("SELECT category, AVG(unit_price) AS avg_price FROM products GROUP BY category;", conn))

# Q16: Count and total revenue per order status
print("\n--- Q16 Result ---")
q16_query = "SELECT status, COUNT(*) AS order_count, SUM(total_amount) AS total_revenue FROM orders GROUP BY status ORDER BY total_revenue DESC;"
display(pd.read_sql_query(q16_query, conn))

# Q17: Max and min product prices per category
print("\n--- Q17 Result ---")
q17_query = "SELECT category, MAX(unit_price) AS max_price, MIN(unit_price) AS min_price FROM products GROUP BY category;"
display(pd.read_sql_query(q17_query, conn))

# Q18: Categories where average unit_price > 2000
print("\n--- Q18 Result ---")
display(pd.read_sql_query("SELECT category, AVG(unit_price) FROM products GROUP BY category HAVING AVG(unit_price) > 2000;", conn))

--- Q13 Result ---


,total_orders
0,10



--- Q14 Result ---


,total_revenue
0,17191



--- Q15 Result ---


...

,status,order_count,total_revenue
0,Delivered,6,17191
1,Shipped,2,13596
2,Cancelled,1,2999
3,Pending,1,1299



--- Q17 Result ---


,category,max_price,min_price
0,Clothing,4599,799
1,Electronics,3499,899
2,Home,1299,599



--- Q18 Result ---


,category,AVG(unit_price)
0,Clothing,2699.0
1,Electronics,2224.0


In [5]:
# Q19: Inner join displaying order details with customer names
print("--- Q19 Result ---")
q19_query = "SELECT o.order_id, o.order_date, c.first_name, c.last_name, o.total_amount FROM orders o INNER JOIN customers c ON o.customer_id = c.customer_id;"
display(pd.read_sql_query(q19_query, conn))

# Q20: Left join listing all customers and their corresponding orders
print("\n--- Q20 Result ---")
q20_query = "SELECT c.customer_id, c.first_name, o.order_id, o.total_amount FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id;"
display(pd.read_sql_query(q20_query, conn))

# Q21: Join across three tables for order item context
print("\n--- Q21 Result ---")
q21_query = """
SELECT oi.order_id, p.product_name, oi.quantity, oi.unit_price, oi.discount_pct
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN products p ON oi.product_id = p.product_id;
"""
display(pd.read_sql_query(q21_query, conn))

# Q22: Join types overview
print("\n--- Q22 Explanation ---")
print("LEFT JOIN preserves all customer rows regardless of order matches. RIGHT JOIN reverses that focus to target all order log metrics.")
print("FULL OUTER JOIN applies when combining complete sets from both tables to find unlinked entries on either side.")

# Q23: Foreign Key evaluation
print("\n--- Q23 Explanation ---")
print("Foreign keys connect orders(customer_id) and order_items(order_id/product_id).")
print("Inserting customer_id 999 fails with a foreign key constraint violation because it does not exist in the primary table.")

--- Q19 Result ---


,order_id,order_date,first_name,last_name,total_amount
0,1001,2024-08-01,Aarav,Sharma,4498
1,1002,2024-08-03,Priya,Patel,799
2,1003,2024-08-05,Rohan,Gupta,7498
3,1004,2024-08-10,Aarav,Sharma,3499
4,1005,2024-08-12,Sneha,Reddy,2999
5,1006,2024-08-15,Vikram,Singh,5898
6,1007,2024-08-18,Ananya,Iyer,1299
7,1008,2024-08-20,Rohan,Gupta,899
8,1009,2024-08-25,Karan,Mehta,6098
9,1010,2024-08-28,Divya,Nair,1598



--- Q20 Result ---


,customer_id,first_name,order_id,total_amount
0,101,Aarav,1001,4498
1,101,Aarav,1004,3499
2,102,Priya,1002,799
3,103,Rohan,1003,7498
4,103,Rohan,1008,899
5,104,Sneha,1005,2999
6,105,Vikram,1006,5898
7,106,Ananya,1007,1299
8,107,Karan,1009,6098
9,108,Divya,1010,1598



--- Q21 Result ---


,order_id,product_name,quantity,unit_price,discount_pct
0,1001,Wireless Earbuds,2,1499,0
1,1001,Laptop Stand,1,899,10
2,1002,Cotton T-Shirt,1,799,0
3,1003,Smart Watch,1,2999,0
4,1003,Running Shoes,1,4599,5
5,1004,Bluetooth Speaker,1,3499,0
6,1005,Smart Watch,1,2999,0
7,1006,Wireless Earbuds,1,1499,10
8,1006,Running Shoes,1,4599,5
9,1007,Bedsheet Set,1,1299,0



--- Q22 Explanation ---
LEFT JOIN preserves all customer rows regardless of order matches. RIGHT JOIN reverses that focus to target all order log metrics.
FULL OUTER JOIN applies when combining complete sets from both tables to find unlinked entries on either side.

--- Q23 Explanation ---
Foreign keys connect orders(customer_id) and order_items(order_id/product_id).
Inserting customer_id 999 fails with a foreign key constraint violation because it does not exist in the primary table.


In [8]:
print("--- Q24 Result ---")
q24_query = """
SELECT product_name, unit_price,
       CASE
           WHEN unit_price < 1000 THEN 'Budget'
           WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
           ELSE 'Premium'
       END AS price_tier
FROM products;
"""
display(pd.read_sql_query(q24_query, conn))

print("\n--- Q25 Result ---")
q25_query = """
SELECT SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS Delivered_Count,
       SUM(CASE WHEN status != 'Delivered' THEN 1 ELSE 0 END) AS Not_Delivered_Count
FROM orders;
"""
display(pd.read_sql_query(q25_query, conn))

print("\n--- Q26 Explanation ---")
print("A - Atomicity: Transactions run fully or not at all (e.g., balance drops only if the recipient gets the money).")
print("C - Consistency: Data changes must follow all schema rules (e.g., account balance cannot become negative if checked).")
print("I - Isolation: Multiple transfers running at the same time don't interfere with each other.")
print("D - Durability: Changes are saved permanently once confirmed, even during power or system crashes.")

print("\n--- Q27 Result ---")
try:
    cursor.execute("BEGIN TRANSACTION;")
    cursor.execute("INSERT OR REPLACE INTO orders VALUES (1011, 102, '2026-05-31', 'Pending', 1598.00);")
    cursor.execute("INSERT OR REPLACE INTO order_items VALUES (5016, 1011, 201, 1, 1499.00, 0);")
    cursor.execute("INSERT OR REPLACE INTO order_items VALUES (5017, 1011, 207, 1, 899.00, 10);")
    cursor.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id IN (201, 207);")
    cursor.execute("COMMIT;")
    print("Transaction committed cleanly.")
except Exception as e:
    cursor.execute("ROLLBACK;")
    print(f"Transaction failed, rolled back: {e}")

conn.close()

--- Q24 Result ---


,product_name,unit_price,price_tier
0,Wireless Earbuds,1499,Mid-Range
1,Cotton T-Shirt,799,Budget
2,Smart Watch,2999,Mid-Range
3,Running Shoes,4599,Premium
4,Bluetooth Speaker,3499,Premium
5,Bedsheet Set,1299,Mid-Range
6,Laptop Stand,899,Budget
7,Cushion Covers (Set),599,Budget



--- Q25 Result ---


,Delivered_Count,Not_Delivered_Count
0,6,4



--- Q26 Explanation ---
A - Atomicity: Transactions run fully or not at all (e.g., balance drops only if the recipient gets the money).
C - Consistency: Data changes must follow all schema rules (e.g., account balance cannot become negative if checked).
I - Isolation: Multiple transfers running at the same time don't interfere with each other.
D - Durability: Changes are saved permanently once confirmed, even during power or system crashes.

--- Q27 Result ---
Transaction committed cleanly.
